# Colab Minimal CFM Overfit Guide

## Recommended: minimal CFM overfit (no Stage 2, no author `results/`)

Run these cells in order:

| Step | Cell | What it does | Time (rough) |
|------|------|--------------|--------------|
| 1 | **1** | Enter project dir | < 1 min |
| 2 | **2** | Install dependencies | 2–5 min |
| 3 | **3** | WandB login (optional) | 1 min |
| 4 | **4** | Check `Data/` exists | < 1 min |
| 5 | **6** | Stage 1 smoke: train GRU 5 epochs | 5–20 min |
| 6 | **8** | Resolve `extractor_path` from latest Stage 1 run | < 1 min |
| 7 | **11** | Stage 3 CFM overfit 100 epochs (1 sample) | 30–90 min |

**Skip:** Cell 5 (one-click), Cell 7 (Stage 2), Cell 9/10 (smoke tests)

## Condition used in Cell 11
`c = [hidden_state, direction, DI]`
- `hidden_state`: from Stage 1 GRU checkpoint (Cell 6 output)
- `direction`, `DI`: from `Data/DamagePattern/label.csv` (no Stage 2 needed)

## If you already have author checkpoints
Skip Cell 6 and run **1 → 2 → 4 → 8 → 11** instead.
Upload:
- `results/HystereticPrediction/HystereticGRU/2024-06-12_18-13-47`

## Success check
- Log shows: `[Overfit mode] using one sample only`
- `train/fm_loss` keeps decreasing
- At epoch 100: `val/psnr` / `val/ssim` should improve


In [ ]:
# === EDIT THIS IF NEEDED ===
PROJECT_DIR = "/content/intership-project"

import os
assert os.path.isdir(PROJECT_DIR), f"Project dir not found: {PROJECT_DIR}"
%cd /content/intership-project
print("Project dir:", PROJECT_DIR)


In [ ]:
# Install dependencies
!pip install -q --upgrade pip
!pip install -q -r requirements.txt


In [ ]:
# Optional: Weights & Biases
USE_WANDB = True  # set False to disable

if USE_WANDB:
    import wandb
    wandb.login()
    WANDB_FLAG = "--use_wandb"
else:
    WANDB_FLAG = ""

print("WANDB_FLAG:", WANDB_FLAG)


In [ ]:
# Data quick check (must exist before training)
required_paths = [
    "Data/Hysteresis/info_w_mander_big.csv",
    "Data/Hysteresis/id_list_small.json",
    "Data/DamagePattern/label.csv",
    "Data/DamagePattern/id_list.json",
]

for p in required_paths:
    print(p, "OK" if os.path.exists(p) else "MISSING")


In [ ]:
# One-click pipeline runner (GAN / CFM / Overfit)
MODE = "overfit"      # "smoke" or "overfit"
BACKEND = "cfm"     # "gan" or "cfm"

wandb_opt = "--with-wandb" if USE_WANDB else ""
!bash scripts/colab_quickstart.sh {wandb_opt} --mode={MODE} --backend={BACKEND}


In [ ]:
# Stage 1 (manual): Hysteretic prediction
!python train_hysteresis.py --architecture HystereticGRU --epoch 5 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# Stage 2 (manual): Damage quantification
!python train_quantification.py --backbone vit_b_16 --epoch 5 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# Resolve Stage 1 extractor path (minimal CFM path: Stage 2 not required)
import glob

AUTHOR_EXTRACTOR = "results/HystereticPrediction/HystereticGRU/2024-06-12_18-13-47"

def latest_run(base_glob):
    runs = sorted(glob.glob(base_glob), key=os.path.getmtime)
    assert runs, f"No runs found for: {base_glob}. Run Cell 6 first."
    return runs[-1]

if os.path.isdir(AUTHOR_EXTRACTOR):
    extractor_path = AUTHOR_EXTRACTOR
    print("Using author extractor:", extractor_path)
else:
    extractor_path = latest_run("results/HystereticPrediction/HystereticGRU/*")
    print("Using latest local Stage 1 run:", extractor_path)


In [ ]:
# Stage 3 (manual): GAN backend smoke
!python train_mixed.py \
  --architecture HystereticCGAN \
  --loss_type SGAN \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning False \
  --direction_conditioning False \
  --add_DI_loss False \
  --image_size 128 \
  --z_dim 100 \
  --G_h_size 128 \
  --D_h_size 128 \
  --SELU False \
  --spectral_D False \
  --no_batch_norm False \
  --lr_schedule warmup_cosine \
  --warmup_epochs 2 \
  --min_lr_ratio 0.1 \
  --epoch 5 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}


In [ ]:
# Stage 3 (manual): CFM ADM backend with Heun solver
!python train_mixed.py \
  --architecture CFMADM \
  --cfm_solver heun \
  --cfm_ode_steps 20 \
  --cfm_model_channels 128 \
  --cfm_num_res_blocks 2 \
  --cfm_cond_dim 512 \
  --cfm_dropout 0.0 \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning True \
  --direction_conditioning True \
  --include_di_conditioning True \
  --add_DI_loss False \
  --image_size 128 \
  --lr_schedule warmup_cosine \
  --warmup_epochs 2 \
  --min_lr_ratio 0.1 \
  --epoch 5 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}


In [ ]:
# Minimal path: Stage 3 CFM overfit (100 epochs, single sample, no Stage 2)
!python train_mixed.py \
  --architecture CFMADM \
  --cfm_solver heun \
  --cfm_ode_steps 20 \
  --extractor_path "{extractor_path}" \
  --data_path id_list.json \
  --latent_conditioning True \
  --direction_conditioning True \
  --include_di_conditioning True \
  --add_DI_loss False \
  --image_size 128 \
  --lr_schedule warmup_cosine \
  --warmup_epochs 10 \
  --min_lr_ratio 0.1 \
  --overfit_single_sample \
  --overfit_sample_index 0 \
  --overfit_repeat_steps 32 \
  --epoch 100 \
  --batch_size 4 \
  --num_workers 0 \
  {WANDB_FLAG}
